# Train: QLoRA SFT on MetaMathQA

Colab front-end for `src/train.py`. Run one ablation config per session.

**Recommended runtime:** A100 (paid). T4 works for the smaller configs but is slow.

1. Set runtime to GPU (T4 free, A100 paid).
2. Run all cells.
3. Adapter saved to `runs/<run_id>/adapter/`. ~5 MB.

## 1. Clone and install

In [ ]:
import os

if not os.path.isdir('finetune-gsm8k'):
    !git clone https://github.com/YuZh98/finetune-gsm8k.git
os.chdir('/content/finetune-gsm8k')
!pip install -q -r requirements.txt

## 2. Sanity check: one tokenized example

Always print one full example before training. Catches chat-template bugs.

In [ ]:
from src.data import load_metamath
from src.utils import load_tokenizer

tok = load_tokenizer(padding_side='right')
ds = load_metamath(8)
print(tok.apply_chat_template(ds[0]['messages'], tokenize=False))

## 3. Pick the ablation row

Set the variables for the run you want. See `src/config.py` for the full matrix.

In [ ]:
from src.config import (
    ABLATION_MATRIX,
    DEFAULT_TARGET_MODULES_ATTN,
    DEFAULT_TARGET_MODULES_ALL,
)

# Index into ABLATION_MATRIX (see src/config.py). Default: 2 = run2_r16 (center config).
RUN_INDEX = 2

cfg = ABLATION_MATRIX[RUN_INDEX]
RUN_ID = cfg.run_id
RANK = cfg.rank
ALPHA = cfg.alpha
LR = cfg.lr
DATA = cfg.data_size

if cfg.target_modules == DEFAULT_TARGET_MODULES_ATTN:
    TARGET = 'attn'
elif cfg.target_modules == DEFAULT_TARGET_MODULES_ALL:
    TARGET = 'all'
else:
    raise ValueError(f'Unknown target_modules for {RUN_ID}: {cfg.target_modules}')

# Not in ABLATION_MATRIX; tune here if needed.
EPOCHS = 1
BATCH_SIZE = 4
GRAD_ACCUM = 4

print(f'{RUN_ID}: rank={RANK} alpha={ALPHA} target={TARGET} lr={LR} data={DATA}')

## 4. Train

In [ ]:
from src.train import train_one_run

train_one_run(
    rank=RANK,
    alpha=ALPHA,
    target=TARGET,
    lr=LR,
    data_size=DATA,
    output_dir=f'./runs/{RUN_ID}',
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
)

## 5. Save adapter to Drive (optional)

Colab wipes runtime storage. Mount Drive to persist.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r ./runs/{RUN_ID} /content/drive/MyDrive/finetune-gsm8k-runs/